# KArSL — Exploratory Analysis

Inspect the extracted directory tree and class/video counts per signer.

In [ ]:
import os, sys
sys.path.append(os.path.abspath('..'))
from utils.config import BASE_DIR, SIGNERS, SPLITS, DRIVE_BASE

In [ ]:
# Walk the dataset tree (top few levels only)
MAX_DEPTH = 4
for root, dirs, files in os.walk(DRIVE_BASE):
    level = root.replace(DRIVE_BASE, '').count(os.sep)
    if level > MAX_DEPTH:
        continue
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')

In [ ]:
# Per-signer, per-split class and video counts
for signer in SIGNERS:
    print(f"\n{'='*40}\n  Signer {signer}\n{'='*40}")
    for split in SPLITS:
        path = os.path.join(BASE_DIR, signer, split, 'extracted')
        if not os.path.exists(path):
            print(f'  {split}: extracted folder not found')
            continue
        total_classes, total_videos = 0, 0
        for cls in sorted(os.listdir(path)):
            cls_path = os.path.join(path, cls)
            if not os.path.isdir(cls_path):
                continue
            total_classes += 1
            total_videos  += len(os.listdir(cls_path))
        print(f'  {split}: {total_classes} classes | {total_videos} videos')

## Sanity-check a single video through the preprocessing pipeline

In [ ]:
import glob, numpy as np
from data_processing.preprocess import read_video, standardize_frames, KeypointExtractor, normalize_frame

# Pick the first available video
sample = None
for signer in SIGNERS:
    for split in SPLITS:
        hits = glob.glob(f'{BASE_DIR}/{signer}/{split}/extracted/*/*.mp4')
        if hits:
            sample = hits[0]; break
    if sample: break
print('Sample video:', sample)

kp = KeypointExtractor()
frames = read_video(sample)
std    = standardize_frames(frames)
kps    = np.stack([kp.extract(f) for f in std])
seq    = np.stack([normalize_frame(f) for f in kps])
print('Sequence shape:', seq.shape, 'dtype:', seq.dtype)